In [ ]:
import os
import subprocess
from glob import glob
import pandas as pd
import numpy as np
import h5py
from scipy.io import mmread
from scipy.sparse import csr_matrix
import shutil 
import gzip
import scipy

# h5 generate

In [ ]:
dataset = "Xe_hsa_kidney_h"
data_dir_root = f"PATH/TO/{dataset}/"
h5_dir_root = f"PATH/TO/{dataset}/drop_out/h5/"
features_path = f"PATH/TO/{dataset}/filtered_feature_bc_matrix/features.tsv.gz"
locs_path = f"{data_dir_root}/locs-raw.tsv"

In [ ]:
locs_raw = pd.read_csv(
            locs_path,
            sep='\t',

            header=0, # if the first row is the header 
            index_col=0
        )
# generate `in_tissue` colunm
locs_raw['in_tissue'] = 1
# generate `array_row` and `array_col` colunm
locs_raw['array_row'] = locs_raw['y']
locs_raw['array_col'] = locs_raw['x']
x = list(locs_raw['x'].values)
y = list(locs_raw['y'].values)
x_keys_sort = sorted(set(x))
y_keys_sort = sorted(set(y))
x_rank_dict = {value: rank for rank, value in enumerate(x_keys_sort)}
y_rank_dict = {value: rank for rank, value in enumerate(y_keys_sort)}
locs_raw = locs_raw.replace({"array_row": y_rank_dict, "array_col": x_rank_dict})
locs_raw["array_row"] = locs_raw["array_row"].astype(int)
locs_raw["array_col"] = locs_raw["array_col"].astype(int)
locs_raw.rename(columns={'x':'pxl_col_in_fullres','y':'pxl_row_in_fullres'},inplace = True)
locs = locs_raw[['in_tissue','array_row','array_col','pxl_row_in_fullres','pxl_col_in_fullres']]
locs.index.name = None
features_raw_data = pd.read_csv(features_path, header=None, sep="\t")
features_raw_data.columns = ['id', 'name', 'feature_type', 'genome']

In [ ]:
drop_out_ratio_all = [0.1, 0.2, 0.3, 0.4, 0.5]
drop_out_ratio_all = [str(element) for element in drop_out_ratio_all]
for dor in drop_out_ratio_all:
    for rep in ["1","2","3"]:
        out_dir = f"{h5_dir_root}/drop_out_{dor}/rep{rep}"
        os.makedirs(out_dir, exist_ok=True)
        os.makedirs(f"{out_dir}/spatial", exist_ok=True)
        exp_data = pd.read_csv(
            f"{data_dir_root}drop_out/cnts/cnts{dor}_rep{rep}.tsv",
            sep='\t',
            header=0,
            index_col='spot'
        )
        # Preserve existing barcodes; prefix numeric labels only when the location table requires it.
        exp_data.index = exp_data.index.astype(str)
        locs.index = locs.index.astype(str)
        if set(exp_data.index) != set(locs.index):
            prefixed = pd.Index(['barcode' + value for value in exp_data.index])
            if set(prefixed) != set(locs.index):
                raise ValueError("Counts and locations contain different barcodes")
            exp_data.index = prefixed
        if not exp_data.index.is_unique or not locs.index.is_unique:
            raise ValueError("Duplicate barcodes are not supported")
        exp_data = exp_data.loc[locs.index]
        locs.to_csv(
            f"{out_dir}/spatial/tissue_positions_list_with_header.csv", 
            sep=',', 
            index=True,
            header=True
        )
        locs.to_csv(
            f"{out_dir}/spatial/tissue_positions_list.csv", 
            sep=',', 
            index=True,
            header=False
        )
        features_use = pd.DataFrame(exp_data.columns)
        features_use.columns = ['name']
        features_raw_data = features_raw_data.drop_duplicates(subset=['name'], keep='first')
        features = features_raw_data.set_index('name').reindex(exp_data.columns).rename_axis('name').reset_index()
        features = features[['id', 'name', 'feature_type', 'genome']]
        if features['id'].isna().any():
            raise ValueError("Missing feature metadata for expression-matrix genes")


        reAnno_count_sparse_mtx = scipy.sparse.csr_matrix(exp_data.T.values)
        with open(os.path.join(out_dir, 'matrix.mtx'), "wb") as matrix_file:
            scipy.io.mmwrite(matrix_file, reAnno_count_sparse_mtx)
        with open(os.path.join(out_dir,'matrix.mtx'),'rb') as mtx_in:
            with gzip.open(os.path.join(out_dir,'matrix.mtx') + '.gz','wb') as mtx_gz:
                shutil.copyfileobj(mtx_in, mtx_gz)
        os.remove(os.path.join(out_dir,'matrix.mtx'))
        ##save barcodes.tsv.gz
        barcodesFile = pd.DataFrame(exp_data.index)
        barcodesFile.to_csv(os.path.join(out_dir,"barcodes.tsv.gz"),sep='\t',header =False,index=False)
        ##save features.tsv.gz
        features.to_csv(os.path.join(out_dir,"features.tsv.gz"),sep='\t',header =False,index=False)
        expression_data = mmread(f"{out_dir}/matrix.mtx.gz").tocsc()
        features = pd.read_csv(f"{out_dir}/features.tsv.gz", header=None, sep="\t")
        # rename columns
        features.columns = ["id", "name", "feature_type","genome"]
        barcodes = pd.read_csv(f"{out_dir}/barcodes.tsv.gz", header=None)
        barcodes.columns = ["barcode"]
        # Load information from files
        barcode_list = barcodes["barcode"].values
        gene_id_list = features["id"].values
        gene_name_list = features["name"].values
        gene_type_list = features["feature_type"].values
        gene_genome_list = features["genome"]
        # generate H5 file
        with h5py.File(f"{out_dir}/filtered_feature_bc_matrix.h5", 'w') as f:
        # Add library_ids to attrs
            f.attrs['library_ids'] = np.array([dataset], dtype='S')
        # Add Matrix group
            group = f.create_group('matrix')
        ## Add data
            group.create_dataset('data', data=expression_data.data)
            group.create_dataset('indices', data=expression_data.indices)
            group.create_dataset('indptr', data=expression_data.indptr)
            group.create_dataset('shape', data=expression_data.shape)
            group.create_dataset('barcodes', data=np.array(barcode_list, dtype='S'))
        ## Add feature group
            feature_group = group.create_group('features')
        ### Add data
            feature_group.create_dataset('id', data=np.array(gene_id_list, dtype='S'))
            feature_group.create_dataset('name', data=np.array(gene_name_list, dtype='S'))
            feature_group.create_dataset('feature_type', data=np.array(gene_type_list, dtype='S'))
            feature_group.create_dataset('genome', data=np.array(gene_genome_list, dtype='S'))
            feature_group.create_dataset('_all_tag_keys', data=np.array([b'genome'], dtype='S'))
        ### Add target group
            target_group = feature_group.create_group('target_sets')
            target_group.create_dataset('AddOn Panel', data=np.arange(len(features), dtype=np.int64))

# Inspect the exported H5


In [ ]:
import h5py
with h5py.File(f"{out_dir}/filtered_feature_bc_matrix.h5","r") as f:
    print(f['matrix'].keys())

In [ ]:
with h5py.File(f"{out_dir}/filtered_feature_bc_matrix.h5", "r") as handle:
    print("Gene x spot shape:", tuple(handle["matrix/shape"][:]))